In [1]:
# ================================================
# HACKER NEWS DATA ANALYSIS
# "What Makes a Post Go Viral?"
# Tool: SQL + BigQuery
# Dataset: bigquery-public-data.hacker_news.full
# Author: Ayush
# ================================================

from google.cloud import bigquery

client = bigquery.Client()
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)

Using Kaggle's public dataset BigQuery integration.


In [2]:
# ================================
# Q1: What are the top 10 viral posts ever?
# ================================

query1 = """
SELECT title, score, `by` AS author
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score IS NOT NULL
AND title IS NOT NULL
AND `by` IS NOT NULL
ORDER BY score DESC
LIMIT 10
"""

q1 = client.query(query1, job_config=safe_config).to_dataframe()
q1

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,score,author
0,Stephen Hawking has died,6015,Cogito
1,A Message to Our Customers,5771,epaga
2,OpenAI's board has fired Sam Altman,5710,davidbarker
3,Backdoor in upstream xz/liblzma leading to SSH...,4549,rkta
4,CrowdStrike Update: Windows Bluescreen and Boo...,4489,BLKNSLVR
5,Steve Jobs has passed away.,4338,patricktomas
6,Bram Moolenaar has died,4310,wufocaculura
7,Mechanical Watch,4298,todsacerdoti
8,YouTube-dl has received a DMCA takedown from RIAA,4240,phantop
9,Don't post generated/AI-edited comments. HN is...,4229,usefulposter


In [3]:
# ================================
# Q2: Which authors post most consistently?
# ================================

query2 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS best_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
HAVING COUNT(*) >= 10
ORDER BY avg_score DESC
LIMIT 10
"""

q2 = client.query(query2, job_config=safe_config).to_dataframe()
q2

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,author,total_posts,avg_score,best_score
0,km,14,425.50,2358
1,_vvdf,19,367.32,1697
2,phiresky,11,356.09,1812
3,atgctg,12,350.75,2360
4,ssiddharth,16,347.75,2210
5,zachrip,11,330.55,3270
6,CathalMullan,11,327.27,1880
7,rushingcreek,18,321.00,1401
8,tinyprojects,18,313.50,1683
9,natdempk,11,311.45,3393


In [4]:
# ================================
# Q3: What % of content is stories vs comments?
# ================================

query3 = """
SELECT type,
       COUNT(*) AS total,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM `bigquery-public-data.hacker_news.full`
WHERE type IS NOT NULL
GROUP BY type
ORDER BY total DESC
"""

q3 = client.query(query3, job_config=safe_config).to_dataframe()
q3

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total,percentage
0,comment,42520344,87.22
1,story,6192401,12.70
2,job,18163,0.04
3,pollopt,15861,0.03
4,poll,2256,0.00


In [5]:
# ================================
# Q4: Which posts scored above average?
# ================================

query4 = """
SELECT title, score, `by` AS author
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score > (
    SELECT AVG(score)
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND score IS NOT NULL
)
AND title IS NOT NULL
AND `by` IS NOT NULL
ORDER BY score DESC
LIMIT 10
"""

q4 = client.query(query4, job_config=safe_config).to_dataframe()
q4

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,score,author
0,Stephen Hawking has died,6015,Cogito
1,A Message to Our Customers,5771,epaga
2,OpenAI's board has fired Sam Altman,5710,davidbarker
3,Backdoor in upstream xz/liblzma leading to SSH...,4549,rkta
4,CrowdStrike Update: Windows Bluescreen and Boo...,4489,BLKNSLVR
5,Steve Jobs has passed away.,4338,patricktomas
6,Bram Moolenaar has died,4310,wufocaculura
7,Mechanical Watch,4298,todsacerdoti
8,YouTube-dl has received a DMCA takedown from RIAA,4240,phantop
9,Don't post generated/AI-edited comments. HN is...,4229,usefulposter


In [6]:
# ================================
# Q5: How does average score vary by post type?
# ================================

query5 = """
SELECT type,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS max_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type IS NOT NULL
AND score IS NOT NULL
GROUP BY type
ORDER BY avg_score DESC
"""

q5 = client.query(query5, job_config=safe_config).to_dataframe()
q5

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total_posts,avg_score,max_score
0,pollopt,15440,39.27,3913
1,poll,2022,26.99,2423
2,story,5928592,13.70,6015
3,job,17440,1.63,220


In [7]:
# ================================
# Q6: Who are top authors by total score?
# ================================

query6 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       SUM(score) AS total_score,
       ROUND(AVG(score), 2) AS avg_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
ORDER BY total_score DESC
LIMIT 10
"""

q6 = client.query(query6, job_config=safe_config).to_dataframe()
q6

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,author,total_posts,total_score,avg_score
0,todsacerdoti,18915,545090,28.82
1,ingve,17266,509032,29.48
2,tosh,25214,428173,16.98
3,pseudolus,20213,394958,19.54
4,rbanffy,37415,363739,9.72
5,Tomte,26563,346034,13.03
6,zdw,6160,312763,50.77
7,thunderbong,16587,287285,17.32
8,bookofjoe,21301,246633,11.58
9,luu,6047,222968,36.87


In [8]:
# ================================
# Q7: Which year had the most posts?
# ================================

query7 = """
SELECT EXTRACT(YEAR FROM TIMESTAMP_SECONDS(time)) AS year,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND time IS NOT NULL
AND score IS NOT NULL
GROUP BY year
ORDER BY year ASC
"""

q7 = client.query(query7, job_config=safe_config).to_dataframe()
q7

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,year,total_posts,avg_score
0,2006,50,5.98
1,2007,22485,5.10
2,2008,70212,6.28
3,2009,111392,8.79
4,2010,179346,10.11
5,2011,292867,9.46
6,2012,311613,9.62
7,2013,308405,11.39
8,2014,291193,11.85
9,2015,326589,11.61


In [9]:
# ================================
# Q8: Categorize posts by popularity tier
# ================================

query8 = """
SELECT 
CASE 
    WHEN score >= 1000 THEN "Legendary"
    WHEN score >= 500  THEN "Viral"
    WHEN score >= 100  THEN "Popular"
    WHEN score >= 10   THEN "Normal"
    ELSE "Low"
END AS category,
COUNT(*) AS total_posts
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score IS NOT NULL
GROUP BY category
ORDER BY total_posts DESC
"""

q8 = client.query(query8, job_config=safe_config).to_dataframe()
q8

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,total_posts
0,Low,5145332
1,Normal,579973
2,Popular,187372
3,Viral,13336
4,Legendary,2579


In [10]:
# ================================
# Q9: Authors with highest consistency (low variance)
# ================================

query9 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS best_score,
       MIN(score) AS worst_score,
       MAX(score) - MIN(score) AS score_range
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
HAVING COUNT(*) >= 15
ORDER BY avg_score DESC
LIMIT 10
"""

q9 = client.query(query9, job_config=safe_config).to_dataframe()
q9

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,author,total_posts,avg_score,best_score,worst_score,score_range
0,_vvdf,19,367.32,1697,12,1685
1,ssiddharth,16,347.75,2210,1,2209
2,rushingcreek,18,321.00,1401,1,1400
3,tinyprojects,18,313.50,1683,1,1682
4,erohead,43,294.19,2843,1,2842
5,granzymes,15,274.33,3089,1,3088
6,Cogito,24,264.92,6015,1,6014
7,coldsnap427,16,263.75,795,1,794
8,rkta,22,257.59,4549,1,4548
9,df07,19,256.89,1734,1,1733


In [11]:
# ================================
# Q10: What % of posts are dead or deleted?
# ================================

query10 = """
SELECT
CASE
    WHEN dead = TRUE THEN "Dead"
    WHEN deleted = TRUE THEN "Deleted"
    ELSE "Active"
END AS status,
COUNT(*) AS total,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
GROUP BY status
ORDER BY total DESC
"""

q10 = client.query(query10, job_config=safe_config).to_dataframe()
q10

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,status,total,percentage
0,Active,4911880,79.32
1,Dead,1280521,20.68


In [ ]:
# ================================
# KEY INSIGHTS
# ================================

print("""
KEY INSIGHTS FROM HACKER NEWS ANALYSIS
=======================================

1. Stephen Hawking's death post = most viral ever (6015 upvotes)
2. Tech controversies & deaths dominate top posts
3. km is the most consistent author (avg 425 score per post)
4. Comments make up ~80% of all content on HN
5. Majority of posts score below 10 — going viral is RARE
6. Post activity peaked around 2016-2017
7. Only ~2% of posts reach "Popular" status or above
8. Dead/deleted posts make up significant portion of data
9. Top authors by total score posted 100s of times consistently
10. Viral posts are unpredictable — even a "Mechanical Watch" went legendary!
""")